# CD-HIT: train vs external sequence diversity (merged set)

Uses the **full** merged training cohort vs PDB-ID–non-overlapping external set.

- Train: `Merged_ECIF_RDKit_train_full.csv` + `PDB_general_kinase_training_2337/`
- External: `Merged_ECIF_RDKit_test_nonoverlapped.csv` + `external_test_nonoverlapped_pairs_5265/`

In [ ]:
from pathlib import Path
import shutil
import subprocess
import pandas as pd
from Bio.PDB import PDBParser, PPBuilder

BASE = Path("/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF")
OUT = BASE / "cdhit_analysis"
OUT.mkdir(exist_ok=True)

TRAIN_CSV = BASE / "Merged_ECIF_RDKit_train_full.csv"
EXT_CSV = BASE / "Merged_ECIF_RDKit_test_nonoverlapped.csv"
TRAIN_DIR = BASE / "PDB_general_kinase_training_2337"
EXT_DIR = BASE / "external_test_nonoverlapped_pairs_5265"

# Local build (already compiled under envs/); falls back to PATH
CDHIT = BASE / "envs" / "cdhit-4.8.1" / "cd-hit"
if not CDHIT.exists():
    CDHIT = Path(shutil.which("cd-hit") or "cd-hit")

CUTOFF = 0.9  # try 0.4 for coarser clustering
print("OUT:", OUT)
print("cd-hit:", CDHIT)

## 1. Extract longest protein chain → FASTA

In [ ]:
parser = PDBParser(QUIET=True)
ppb = PPBuilder()


def find_pdb(root, cid, train=True):
    for name in (cid, cid.lower(), cid.upper()):
        d = root / name
        if not d.is_dir():
            continue
        if train:
            hits = list(d.glob("*_protein.pdb"))
        else:
            hits = list(d.glob("*_protein_receptor.pdb")) or list(d.glob("*_protein.pdb"))
        if hits:
            return hits[0]
    return None


def longest_seq(pdb_path):
    struct = parser.get_structure("x", str(pdb_path))
    seqs = [str(pp.get_sequence()) for pp in ppb.build_peptides(struct)]
    return max(seqs, key=len) if seqs else ""


train_ids = pd.read_csv(TRAIN_CSV, usecols=["Complex_ID"])["Complex_ID"].astype(str)
ext_ids = pd.read_csv(EXT_CSV, usecols=["Complex_ID"])["Complex_ID"].astype(str)

fasta = OUT / "train_ext_proteins.fasta"
missing = []
n = 0

with open(fasta, "w") as f:
    for cid in train_ids:
        p = find_pdb(TRAIN_DIR, cid, train=True)
        if p is None:
            missing.append({"split": "train", "Complex_ID": cid, "reason": "no_pdb"})
            continue
        seq = longest_seq(p)
        if len(seq) < 30:
            missing.append({"split": "train", "Complex_ID": cid, "reason": "short_seq"})
            continue
        f.write(f">train|{cid}\n{seq}\n")
        n += 1

    for cid in ext_ids:
        p = find_pdb(EXT_DIR, cid, train=False)
        if p is None:
            missing.append({"split": "ext", "Complex_ID": cid, "reason": "no_pdb"})
            continue
        seq = longest_seq(p)
        if len(seq) < 30:
            missing.append({"split": "ext", "Complex_ID": cid, "reason": "short_seq"})
            continue
        f.write(f">ext|{cid}\n{seq}\n")
        n += 1

pd.DataFrame(missing).to_csv(OUT / "fasta_missing.csv", index=False)
print(f"Wrote {n} sequences → {fasta}")
print(f"Missing/short: {len(missing)} (see fasta_missing.csv)")

## 2. Run CD-HIT

In [ ]:
tag = f"clusters_{int(CUTOFF*100)}"
out_pref = OUT / tag
cmd = [
    str(CDHIT),
    "-i", str(fasta),
    "-o", str(out_pref),
    "-c", str(CUTOFF),
    "-n", "5" if CUTOFF >= 0.7 else "2",
    "-M", "8000",
    "-T", "4",
]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-1500:] if r.stdout else "")
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit(f"cd-hit failed (exit {r.returncode})")
print("Done:", out_pref, str(out_pref) + ".clstr")

## 3. Summarize train vs external clusters

In [ ]:
clstr = OUT / f"clusters_{int(CUTOFF*100)}.clstr"
rows, cluster_id = [], -1

with open(clstr) as f:
    for line in f:
        line = line.strip()
        if line.startswith(">Cluster"):
            cluster_id = int(line.split()[1])
        elif ">" in line:
            tag = line.split(">")[1].split("...")[0]  # train|3r02
            split, cid = tag.split("|", 1)
            rows.append({"cluster": cluster_id, "split": split, "Complex_ID": cid})

df = pd.DataFrame(rows)
df.to_csv(OUT / f"cluster_membership_{int(CUTOFF*100)}.csv", index=False)

train_c = set(df.loc[df.split == "train", "cluster"])
ext_c = set(df.loc[df.split == "ext", "cluster"])
shared = train_c & ext_c

ext_df = df[df.split == "ext"].copy()
ext_df["seen_in_train"] = ext_df["cluster"].isin(train_c)

summary = pd.DataFrame({
    "metric": [
        "n_train_seqs",
        "n_ext_seqs",
        "n_clusters_train",
        "n_clusters_ext",
        "n_shared_clusters",
        "n_only_ext_clusters",
        "ext_in_shared_clusters",
        "ext_in_novel_clusters",
        "pct_ext_in_shared_clusters",
    ],
    "value": [
        int((df.split == "train").sum()),
        int((df.split == "ext").sum()),
        len(train_c),
        len(ext_c),
        len(shared),
        len(ext_c - train_c),
        int(ext_df.seen_in_train.sum()),
        int((~ext_df.seen_in_train).sum()),
        round(100 * ext_df.seen_in_train.mean(), 1),
    ],
})
summary.to_csv(OUT / f"cdhit_summary_{int(CUTOFF*100)}.csv", index=False)
display(summary)
print(f"\nAt {int(CUTOFF*100)}% identity: {len(shared)} shared clusters; "
      f"{ext_df.seen_in_train.mean()*100:.1f}% of external complexes fall in train-seen clusters.")